Build dataset. Here is an example for SPICE.

Dataset preprocessing may take minutes to hours depending on size.

In [ ]:
import os
import re
import h5py
import torch
import numpy as np
import resff
from resff.units import *
from openff.toolkit.topology import Molecule
from simtk import unit
from simtk.unit import Quantity
from collections import defaultdict

SOURCE_H5 = 'SPICE-1.1.4.hdf5'
OUTPUT_BASE = 'ResFF/data/spice'
os.makedirs(OUTPUT_BASE, exist_ok=True)

def clean_subset_name(name):
   
    # Remove 'v' followed by digits and dots (e.g., v1.1, v2.0.1)
    name = re.sub(r'\s+v\d+(\.\d+)*', '', name)
    # Replace spaces with underscores
    name = name.replace(' ', '_')
    return name

subset_to_groups = defaultdict(list)

with h5py.File(SOURCE_H5, 'r') as f_in:
    for group_name in f_in.keys():
        subsets = f_in[group_name]['subset'][:]
        for sub in subsets:
            decoded_name = sub.decode('utf-8') if isinstance(sub, bytes) else str(sub)
            subset_to_groups[decoded_name].append(group_name)

    for original_name, groups in subset_to_groups.items():
        cleaned_name = clean_subset_name(original_name)
        new_h5_path = os.path.join(OUTPUT_BASE, f"{cleaned_name}.h5")
        
        print(f"Extracting [{original_name}] to [{cleaned_name}.h5]...")
        
        with h5py.File(new_h5_path, 'w') as f_out:
            for g_name in groups:
                # Copy the entire group structure
                f_in.copy(g_name, f_out)



print("Processing extracted files into resff graphs...")
h5_files = [f for f in os.listdir(OUTPUT_BASE) if f.endswith('.h5')]
total_conf_count = 0

for h5_filename in h5_files:
    subset_slug = h5_filename.replace('.h5', '')
    h5_path = os.path.join(OUTPUT_BASE, h5_filename)
    save_subdir = os.path.join(OUTPUT_BASE, subset_slug)
    os.makedirs(save_subdir, exist_ok=True)
    
    print(f"Processing dataset: {subset_slug}")
    with h5py.File(h5_path, 'r') as h5file:
        for smiles_key in h5file.keys():
            record = h5file[smiles_key]
            
            try:
                smi = record["smiles"][0].decode('UTF-8')
            except Exception:
                continue

            # Skip molecules containing specific metal ions
            if any(el in smi for el in ['Li', 'Na', 'Mg', 'K', 'Ca']):
                continue

            # Generate OpenFF Molecule and resff Graph
            try:
                offmol = Molecule.from_mapped_smiles(smi, allow_undefined_stereo=True)
                g = resff.Graph(offmol)
            except Exception:
                continue

            energy = record["dft_total_energy"][()]
        
            u_ref_prime = record["dft_total_gradient"][()].transpose(1, 0, 2)
            xyz = record["conformations"][()].transpose(1, 0, 2)

            energy_min = np.min(energy)
            valid_mask = (energy - energy_min) <= 0.1
            
            energy = energy[valid_mask]
            u_ref_prime = u_ref_prime[:, valid_mask, :]
            xyz = xyz[:, valid_mask, :]

            g.nodes["g"].data["u_ref"] = torch.tensor(
                Quantity(energy, resff.units.HARTREE_PER_PARTICLE).value_in_unit(resff.units.ENERGY_UNIT),
                dtype=torch.get_default_dtype()
            )[None, :]

            g.nodes["n1"].data["xyz"] = torch.tensor(
                Quantity(xyz, unit.bohr).value_in_unit(resff.units.DISTANCE_UNIT),
                requires_grad=True,
                dtype=torch.get_default_dtype()
            )

            g.nodes["n1"].data["u_ref_prime"] = torch.tensor(
                Quantity(u_ref_prime, resff.units.HARTREE_PER_PARTICLE / unit.bohr).value_in_unit(resff.units.FORCE_UNIT),
                dtype=torch.get_default_dtype()
            )

            # Save the individual graph
            molecule_idx += 1
            graph_save_path = os.path.join(save_subdir, str(molecule_idx))
            resff.Graph.save(g, graph_save_path)

print(f"\nProcessing Complete.")